# Data Wrangling — Tiki E-commerce (2 bộ)

Nguồn: [Vietnamese Tiki E-commerce Dataset (Kaggle)](https://www.kaggle.com/datasets/michaelminhpham/vietnamese-tiki-e-commerce-dataset)

1. `vietnamese_tiki_products_men_shoes.csv` — giày nam  
2. `vietnamese_tiki_products_women_shoes.csv` — giày nữ  

Mục tiêu gốc của bộ dữ liệu: dự đoán `quantity_sold`.

Mỗi bước có **1 dòng mẫu**. Dòng `# TODO` em làm tương tự (đổi tên cột / file).

## Download data and explore

Luôn bắt đầu bằng **nhìn dữ liệu**, đừng nhảy vào `fillna`. Hỏi: bao nhiêu dòng? cột nào object? giá trị lạ (`...`, tab, rỗng)?

| Cột | Ý nghĩa |
|---|---|
| `id` | Mã sản phẩm Tiki |
| `name` | Tên sản phẩm |
| `description` | Mô tả (có thể rỗng / `...`) |
| `original_price` | Giá gốc (VND) |
| `price` | Giá hiện tại (VND) |
| `fulfillment_type` | Hình thức giao: `dropship`, `tiki_delivery`, `seller_delivery` |
| `brand` | Thương hiệu (nhiều `OEM`, đôi khi dính tab `\\tOEM`) |
| `review_count` | Số đánh giá |
| `rating_average` | Điểm trung bình (0–5; 0 thường = chưa có đánh giá) |
| `favourite_count` | Số lượt yêu thích |
| `pay_later` | Có trả sau hay không |
| `current_seller` | Tên shop |
| `date_created` | Số ngày từ lần cập nhật |
| `number_of_images` | Số ảnh |
| `vnd_cashback` | Số tiền hoàn (VND) |
| `has_video` | Có video hay không |
| `category` | Danh mục (`Root` = chưa gán danh mục rõ) |
| `quantity_sold` | Tổng số đã bán (biến mục tiêu) |

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

path_men = "tiki_data/vietnamese_tiki_products_men_shoes.csv"
path_women = "tiki_data/vietnamese_tiki_products_women_shoes.csv"

df_men = pd.read_csv(path_men)  # mẫu: đọc file nam
# TODO: đọc file nữ vào df_women (giống dòng trên, đổi path_women)
df_women = pd.read_csv(path_women)

print("Men shoes:", df_men.shape)  # mẫu: in số dòng, số cột
# TODO: in shape của df_women
print("Women shoes: ", df_women.shape)

df_men.head()  # mẫu: 5 dòng đầu
# TODO: xem 5 dòng đầu df_women
df_women.head()

Men shoes: (5745, 19)
Women shoes:  (5919, 19)


,Unnamed: 0,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold
0,0,252071637,"Hộp dép, hộp đựng dép hm - màu cam kich thước ...","Hộp dép, hộp đựng dép hm - màu cam, kíc...",3000,3000,dropship,OEM,0,0.0,0,False,Kho Chuyên Sỉ lẻ Dép,149,7,0,False,Root,0
1,1,208439152,Dép xốp quai ngang đi trong nhà- dép xốp khách...,"Dép xốp quai ngang, đi trong nhà/Khách sạn/Đ...",14200,14200,dropship,OEM,0,0.0,0,False,TT8695,277,6,0,False,Root,0
2,2,105883794,dép tổ ông huyền thoại,Thiết kế đặc biệtChống trơn trượtKhông ngậm nư...,9900,9900,dropship,OEM,3,5.0,0,False,Rumyh Fashion Style,819,4,267,False,Dép quai ngang,8
3,3,252864898,Sen guốc trơn lá lớn_Ảnh thật,"Chào bạn, cảm ơn bạn đã ghé thăm shop ạ...",15000,15000,dropship,OEM,0,0.0,0,False,BOYBON Garden,139,1,0,False,Root,0
4,4,252865522,Cây guốc sao trơn,Cây hawothia guốc sao trơn cây đang con...,13000,13000,dropship,OEM,0,0.0,0,False,BOYBON Garden,139,1,0,False,Root,0


---
# Bộ 1 — Giày nam

## 1. Xử lý giá trị thiếu (Missing Values)

- Chuỗi rỗng / `...` / `"nan"` → `NaN`.
- `fulfillment_type`, `brand`: cột phân loại → **mode**.
- `description`: điền `""` (không xoá dòng).
- `quantity_sold`: biến mục tiêu → `.dropna(subset=["quantity_sold"])`.

In [ ]:
df = df_men.copy()  # mẫu: làm việc trên bản sao

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])  # mẫu: xoá cột index thừa

# TODO: xoá trùng theo id rồi reset_index — gợi ý: drop_duplicates(subset=["id"])
df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].astype(str).str.strip()  # mẫu: cắt khoảng trắng 2 đầu
    # TODO: trong vòng for, replace {"nan": np.nan, "": np.nan, "...": np.nan} cho df[c]
    df[c] = df[c].replace({"nan": np.nan, "": np.nan, "...": np.nan})

df["brand"] = df["brand"].str.replace("\t", "", regex=False)  # mẫu: xoá tab trong brand

print(df.isnull().sum())  # mẫu: đếm missing trước xử lý

df["fulfillment_type"] = df["fulfillment_type"].fillna(
    df["fulfillment_type"].mode(dropna=True)[0]
)  # mẫu: cột chữ → điền mode

# TODO: fillna mode cho cột brand (copy mẫu, đổi tên cột)
df["brand"] = df["brand"].fillna(df["brand"].mode(dropna=True)[0])

# TODO: df["description"] fillna bằng ""
df["description"] = df["description"].fillna("")

# TODO: dropna subset=["quantity_sold"] rồi reset_index

# TODO: in lại isnull().sum() sau khi xử lý

## 2. Sửa định dạng dữ liệu (Correct Data Format)

In [ ]:
print(df.dtypes)  # mẫu: xem kiểu hiện tại

df["price"] = df["price"].astype(int)  # mẫu: ép price về int
# TODO: ép original_price, quantity_sold, review_count về int (giống dòng trên)
# TODO: ép rating_average về float

# TODO: print dtypes các cột vừa ép

## 3. Chuẩn hoá dữ liệu (Data Standardization)

Giá đang là **VND**. Đổi sang **nghìn đồng** cho dễ đọc, và tính **% giảm giá** so với giá gốc.

In [ ]:
df["price_nghin"] = df["price"] / 1000  # mẫu: VND → nghìn đồng
# TODO: cột original_price_nghin = original_price / 1000

# TODO: cột discount_pct = (original_price - price) / original_price * 100
#       dùng np.where(original_price > 0, công_thức, 0)

df[["price", "price_nghin"]].head()  # mẫu: xem kết quả
# TODO: head thêm original_price và discount_pct

## 4. Chuẩn hoá phạm vi giá trị (Data Normalization)

`price` và `quantity_sold` lệch thang đo rất mạnh → `x / x.max()` về 0–1.

In [ ]:
df["price_normalized"] = df["price"] / df["price"].max()  # mẫu: chia max → 0–1
# TODO: quantity_sold_normalized = quantity_sold / max
# TODO: review_count_normalized = review_count / max

df[["price", "price_normalized"]].head()  # mẫu
# TODO: head thêm quantity_sold và cột normalize tương ứng

## 5. Phân nhóm (Binning)

Chia giá thành Low / Medium / High theo ngưỡng VND thực tế (không dùng min–max đều vì giá max rất lớn, hầu hết sản phẩm sẽ rơi vào Low).

In [ ]:
price_bins = [0, 100_000, 500_000, df["price"].max() + 1]  # mẫu: 3 khoảng giá

df["price_binned"] = pd.cut(
    df["price"], bins=price_bins, labels=["Low", "Medium", "High"], include_lowest=True
)  # mẫu: gán nhãn Low / Medium / High

print(df["price_binned"].value_counts())  # mẫu: đếm mỗi nhóm

# TODO: vẽ bar — gợi ý: df["price_binned"].value_counts().sort_index().plot(kind="bar")
# TODO: plt.xlabel / ylabel / title rồi plt.show()

df[["price", "price_binned"]].head()

## 6. Biến chỉ thị / Biến giả (Dummy Variable)

`fulfillment_type` (chữ) → mỗi hình thức giao một cột 0/1 bằng `pd.get_dummies()`, rồi xoá cột gốc.

In [ ]:
dummy_ff = pd.get_dummies(df["fulfillment_type"], prefix="fulfillment", dtype=int)  # mẫu: cột 0/1

# TODO: gộp dummy vào df — pd.concat([df, dummy_ff], axis=1)

# TODO: xoá cột gốc — df.drop(columns=["fulfillment_type"])

df.head()  # mẫu: kiểm tra bảng

## Check and save

Trước khi `to_csv`:

* `isnull().sum()` còn 0 (hoặc đúng chỗ cố ý để thiếu)
* `price`, `quantity_sold` là số
* có cột mới: `price_nghin`, `*_normalized`, `price_binned`, `fulfillment_*`

`index=False` để file không thêm cột index 0,1,2,…


In [ ]:
df.to_csv("tiki_men_shoes_clean.csv", index=False)  # mẫu: lưu file nam
print(df.shape)

# TODO: gán df_men_clean = df để giữ bản sạch bộ nam

---
---
# Bộ 2 — Giày nữ

Làm **giống bộ nam**, đổi `df_men` → `df_women`. Copy mẫu ở trên, sửa tên biến. Kết thúc bằng **Check and save** (`to_csv`, `index=False`).


In [ ]:
df = df_women.copy()  # mẫu: bắt đầu từ file nữ

# TODO: 

# TODO: strip + replace nan/""/"..." trên cột object; xoá \t ở brand

print(df.isnull().sum())  # mẫu

# TODO: fillna mode fulfillment_type (đã có mẫu bộ nam) và brand; description → ""

# TODO: dropna quantity_sold, reset_index, in isnull().sum() lần nữa

In [ ]:
df["price"] = df["price"].astype(int)  # mẫu bước 2
# TODO: ép các cột số còn lại (như bộ nam)

df["price_nghin"] = df["price"] / 1000  # mẫu bước 3
# TODO: original_price_nghin và discount_pct

df["price_normalized"] = df["price"] / df["price"].max()  # mẫu bước 4
# TODO: normalize quantity_sold và review_count

# TODO: bước 5 — pd.cut + value_counts + plot (copy mẫu bộ nam, đổi title Women)

dummy_ff = pd.get_dummies(df["fulfillment_type"], prefix="fulfillment", dtype=int)  # mẫu bước 6
# TODO: concat + drop fulfillment_type

# TODO: to_csv tiki_women_shoes_clean.csv; print shape; df.head()